# Partie 2 - Modélisation ML (Prédiction des prix)

**Objectif** : Entraîner un modèle XGBoost et le sauvegarder pour l'API.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import joblib
import os

In [ ]:
# 1. Chargement des données
data = pd.read_csv('../Data/get_around_pricing_project.csv')
data.drop('Unnamed: 0', axis=1, inplace=True)
data.head()

In [ ]:
# 2. Nettoyage
data = data[(data['engine_power'] >= 50) & (data['engine_power'] <= 350)]
data = data[(data['mileage'] >= 0) & (data['mileage'] < 500000)]

for col in ['mileage', 'engine_power']:
    mean, std = data[col].mean(), data[col].std()
    data = data[(data[col] >= mean - 3*std) & (data[col] <= mean + 3*std)]

In [ ]:
# 3. Feature Engineering
numeric_init = ['mileage', 'engine_power']
numericals = numeric_init.copy()
for col in numeric_init:
    data[f'{col}_2'] = data[col] ** 2
    data[f'{col}_inv'] = 1 / (data[col] + 1e-6)
    numericals.append(f'{col}_2')
    numericals.append(f'{col}_inv')

X = data.drop('rental_price_per_day', axis=1)
y = data['rental_price_per_day']

In [ ]:
# 4. Prétraitement
categoricals = list(X.columns.drop(numericals))
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numericals),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categoricals)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 5. Pipeline et entraînement
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        eta=0.1, n_estimators=170, max_depth=6,
        min_child_weight=2, gamma=0.8, colsample_bytree=0.4,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

# Évaluation
r2 = r2_score(y_test, pipeline.predict(X_test))
print(f"R² sur le test : {r2:.4f}")

In [ ]:
# 6. Sauvegarde du modèle
os.makedirs('../models', exist_ok=True)
joblib.dump(pipeline, '../models/pipeline_xgboost.joblib')
print("✅ Modèle sauvegardé dans 'models/pipeline_xgboost.joblib'")